# 06 – Gentzkowin leave-out-mittari (vertailu luokittelijaan)

**Tehtävä:** puolueiden erot lasketaan kirjallisuuden vakiomenetelmällä, Gentzkowin, Shapiron ja Taddyn (2019) *leave-out*-estimaattorilla (*leave-out estimator*). Käsitteet on koottu muistion 05 taulukkoon. Simola ym. käyttävät samaa mittaria eduskunnalle 1907–2018. Jos mittari ja luokittelija (muistio 05) antavat saman kuvan, tulos ei riipu menetelmän valinnasta.

**Syöte:** `Data/processed/counts.npz`, `counts_index.parquet` ja `vocab.txt` (muistio 04)
**Tulokset** kansioon `results/`:
- `leaveout_party_pairs_term.csv`, `leaveout_party_pairs_year.csv`: 28 puolueparia kausittain ja vuosittain
- `leaveout_gov_term.csv`, `leaveout_gov_year.csv`: hallitus vs. oppositio

**Kesto:** noin 10 minuuttia. Valmiit tulokset luetaan kansiosta `results/`; uudelleenlaskenta asetuksella `UUDELLEEN = True`.

## Mittari

Puolueeton kuulija tuntee kummankin puolueen sanankäytön. Hän kuulee yhden satunnaisen sanaparin edustajan puheesta ja arvaa puolueen (kummallekin puolueelle sama ennakkotodennäköisyys). **π (*partisanship*) on todennäköisyys, että arvaus osuu oikeaan;** π = 0,5 tarkoittaa, ettei puolueilla ole eroa.

Puolueille A ja B:

$$\pi = \tfrac12 \cdot \frac{1}{|A|}\sum_{i \in A} q_i \cdot \rho_{-i} \; + \; \tfrac12 \cdot \frac{1}{|B|}\sum_{i \in B} q_i \cdot (1-\rho_{-i})$$

- $q_i$ = edustajan $i$ sanaparien osuudet hänen puheestaan
- $\rho_{-i}$ = kullekin sanaparille todennäköisyys, että sen sanoja on A:n edustaja: $q^A/(q^A+q^B)$

**Leave-out:** puolueen sanankäyttö ($q^A$ tai $q^B$) lasketaan **ilman edustajaa $i$**. Muuten edustajan harvinaiset sanaparit näyttäisivät puolueelle tyypillisiltä vain siksi, että hän on itse sanonut ne. Taustaa: π on Bayesin kaavan mukainen posterioritodennäköisyys ([SfDS 8.3.1](https://lacerbi.github.io/stats-for-ds-website/chapters/08-bayesian-inference-decision-theory.html#the-engine-bayes-theorem-for-inference)), ja *plug-in*-estimaattori ([SfDS 4.4.2](https://lacerbi.github.io/stats-for-ds-website/chapters/04-nonparametric-bootstrap.html#estimating-functionals-the-plug-in-estimator)) voi olla harhainen ([SfDS 3.6.4](https://lacerbi.github.io/stats-for-ds-website/chapters/03-convergence-inference.html#how-to-evaluate-estimators)). Gentzkow ym. osoittavat, että tällainen *plug-in*-mittari yliarvioi erot moninkertaisesti ja voi tuottaa näennäisiä trendejä (s. 1314–1315 ja 1319–1320). Kaava on heidän yhtälönsä leave-out-estimaattorista (s. 1315).

## Menettely

- **Yksikkö** on edustaja × kausi (tai edustaja × vuosi): edustajan kaikki puheet lasketaan yhteen.
- **Vain sanaparit** kuten artikkeleissa (12 741 kpl). Gentzkow ym. (s. 1316) suosittelevat lisäksi penalisoitua estimaattoria, jossa vakioidaan alue ja muut taustatekijät; sitä emme tee, joten leave-out-mittari on vertailukohta eikä päätulos.
- **Satunnaissarja** (*random series*, Gentzkow ym. s. 1318): parin puolueiden nimikkeet sekoitetaan edustajien kesken ja mittari lasketaan uudelleen. Tuloksen pitäisi olla lähellä 0,5; estimaattori on hieman alaspäin harhainen, joten käytännössä se on noin 0,49–0,50.
- **Epävarmuus** (osaotanta, *subsampling*, Gentzkow ym. s. 1321): 30 kertaa arvotaan 80 % kummankin puolueen edustajista; väli on näiden 10.–90. persentiili.

In [ ]:
import os
import sys
from itertools import combinations

import numpy as np
import pandas as pd
import scipy.sparse as sp

sys.path.insert(0, "../src")
import config

KANSIO = "../Data/processed/"
TULOKSET = "../results/"
OSAOTOKSIA = 30
OSAOTOKSEN_OSUUS = 0.8
UUDELLEEN = False      # True: laske uudelleen, vaikka tulokset ovat jo kansiossa results/

X = sp.load_npz(KANSIO + "counts.npz").tocsr()
indeksi = pd.read_parquet(KANSIO + "counts_index.parquet")
with open(KANSIO + "vocab.txt", encoding="utf-8") as f:
    sanasto = np.array(f.read().split("\n")[:-1])

# vain sanaparit
on_pari = np.array(["_" in fraasi for fraasi in sanasto])
X = X[:, on_pari]
print("Sanapareja:", X.shape[1])

indeksi["year"] = indeksi["start_time"].dt.year.astype(str)
indeksi["asema"] = indeksi["in_government"].map({True: "hallitus", False: "oppositio"})

Sanapareja: 12739


In [2]:
def puhujakohtaiset_maarat(indeksi, X, jakso, ryhma):
    """Laskee puheiden sanaparit yhteen: yksi rivi per jakso (kausi/vuosi) × edustaja × ryhmä."""
    avain = indeksi[jakso].astype(str) + "|" + indeksi["person_id"].astype(str) + "|" + indeksi[ryhma].astype(str)
    koodit, rivi_indeksi = np.unique(avain, return_inverse=True)
    yhdistys = sp.csr_matrix((np.ones(len(indeksi)), (rivi_indeksi, np.arange(len(indeksi)))),
                             shape=(len(koodit), len(indeksi)))
    maarat = (yhdistys @ X).tocsr()
    tiedot = pd.DataFrame([k.split("|") for k in koodit], columns=[jakso, "person_id", ryhma])
    return tiedot, maarat


def ryhman_osuus(C, oma_summa, oma_yhteensa, toisen_q):
    """Keskiarvo q_i · rho_{-i} ryhmän edustajien yli.

    C = ryhmän edustajien sanaparimäärät (edustajat × sanaparit).
    Ryhmän sanankäyttö lasketaan jokaiselle edustajalle ilman häntä itseään (leave-out)."""
    m = C.sum(axis=1, keepdims=True)                   # edustajan sanapareja yhteensä
    oma_q = (oma_summa - C) / (oma_yhteensa - m)       # ryhmän osuudet ilman edustajaa i
    nimittaja = oma_q + toisen_q
    with np.errstate(invalid="ignore", divide="ignore"):
        rho = np.where(nimittaja > 0, oma_q / nimittaja, 0.5)
    q_i = C / m                                        # edustajan omat osuudet
    return np.mean(np.sum(q_i * rho, axis=1))


def leave_out_pi(Ca, Cb):
    """Gentzkowin ym. leave-out-estimaatti kahdelle ryhmälle."""
    Ca = np.asarray(Ca.todense(), dtype=float)
    Cb = np.asarray(Cb.todense(), dtype=float)
    Ca = Ca[Ca.sum(axis=1) > 0]                        # pois edustajat, joilla ei yhtään sanaparia
    Cb = Cb[Cb.sum(axis=1) > 0]
    Sa = Ca.sum(axis=0)
    Sb = Cb.sum(axis=0)
    qa = Sa / Sa.sum()
    qb = Sb / Sb.sum()
    return 0.5 * ryhman_osuus(Ca, Sa, Sa.sum(), qb) + 0.5 * ryhman_osuus(Cb, Sb, Sb.sum(), qa)

In [3]:
def arvioi_parit(tiedot, maarat, jakso, ryhma, parit, rng):
    """Estimaatti, satunnaissarja ja osaotosväli jokaiselle jaksolle ja parille."""
    rivit = []
    for arvo in sorted(tiedot[jakso].unique()):
        jaksossa = tiedot[jakso] == arvo
        for a, b in parit:
            ia = np.where(jaksossa & (tiedot[ryhma] == a))[0]
            ib = np.where(jaksossa & (tiedot[ryhma] == b))[0]
            if len(ia) < 3 or len(ib) < 3:
                continue
            estimaatti = leave_out_pi(maarat[ia], maarat[ib])

            # satunnaissarja: nimikkeet sekoitetaan parin edustajien kesken
            sekoitettu = rng.permutation(np.concatenate([ia, ib]))
            satunnainen = leave_out_pi(maarat[sekoitettu[:len(ia)]], maarat[sekoitettu[len(ia):]])

            # osaotokset: 80 % kummankin ryhmän edustajista
            osaotokset = []
            for _ in range(OSAOTOKSIA):
                sa = rng.choice(ia, size=max(2, int(OSAOTOKSEN_OSUUS * len(ia))), replace=False)
                sb = rng.choice(ib, size=max(2, int(OSAOTOKSEN_OSUUS * len(ib))), replace=False)
                osaotokset.append(leave_out_pi(maarat[sa], maarat[sb]))

            rivit.append({jakso: arvo, "a": a, "b": b, "pi": estimaatti, "pi_random": satunnainen,
                          "sub_lo": np.percentile(osaotokset, 10), "sub_hi": np.percentile(osaotokset, 90),
                          "n_a": len(ia), "n_b": len(ib)})
        print(arvo, "valmis")
    return pd.DataFrame(rivit)

## Ajo

Puolueparit muodostetaan kaikilla kausilla mukana olevista 8 puolueesta (28 paria), ensin kausittain ja sitten vuosittain.

In [4]:
rng = np.random.default_rng(7)
puolueet = list(config.MAIN_PARTIES)
puolueparit = list(combinations(puolueet, 2))
mukana = indeksi["party"].isin(puolueet).values

tiedostot = {nimi: TULOKSET + "leaveout_" + nimi + ".csv"
             for nimi in ["party_pairs_term", "gov_term", "party_pairs_year", "gov_year"]}

if not UUDELLEEN and all(os.path.exists(t) for t in tiedostot.values()):
    tulokset = {nimi: pd.read_csv(tiedosto) for nimi, tiedosto in tiedostot.items()}
    print("Tulokset on jo laskettu, luetaan tiedostoista.")
else:
    tulokset = {}
    for jakso in ["term", "year"]:
        tiedot, maarat = puhujakohtaiset_maarat(indeksi[mukana], X[mukana], jakso, "party")
        tulokset["party_pairs_" + jakso] = arvioi_parit(tiedot, maarat, jakso, "party", puolueparit, rng)

        tiedot, maarat = puhujakohtaiset_maarat(indeksi, X, jakso, "asema")
        tulokset["gov_" + jakso] = arvioi_parit(tiedot, maarat, jakso, "asema", [("hallitus", "oppositio")], rng)

    for nimi, taulu in tulokset.items():
        taulu.to_csv(tiedostot[nimi], index=False)
    print("Tallennettu.")

2015-2019 valmis
2019-2023 valmis
2023-2027 valmis
2015-2019 valmis
2019-2023 valmis
2023-2027 valmis
2015 valmis
2016 valmis
2017 valmis
2018 valmis
2019 valmis
2020 valmis
2021 valmis
2022 valmis
2023 valmis
2024 valmis
2025 valmis
2026 valmis
2015 valmis
2016 valmis
2017 valmis
2018 valmis
2019 valmis
2020 valmis
2021 valmis
2022 valmis
2023 valmis
2024 valmis
2025 valmis
2026 valmis
Tallennettu.


## Tulokset lyhyesti

Tulkinta ja vertailu luokittelijaan ovat muistiossa `07_tulokset`.

In [5]:
kausittain = tulokset["party_pairs_term"].groupby("term")[["pi", "pi_random", "sub_lo", "sub_hi"]].mean()
kausittain["hallitus vs. oppositio"] = tulokset["gov_term"].set_index("term")["pi"]
kausittain.round(4)

,pi,pi_random,sub_lo,sub_hi,hallitus vs. oppositio
term,,,,,
2015-2019,0.5196,0.4948,0.5131,0.5205,0.5141
2019-2023,0.5246,0.4962,0.5161,0.5269,0.5199
2023-2027,0.5317,0.4942,0.5240,0.5340,0.5211


In [6]:
parit_2023 = tulokset["party_pairs_term"]
parit_2023 = parit_2023[parit_2023["term"] == "2023-2027"].sort_values("pi", ascending=False)
print("Kauden 2023–27 eniten ja vähiten eroavat parit:")
print(parit_2023[["a", "b", "pi", "sub_lo", "sub_hi"]].round(4).head(5).to_string(index=False))
print("...")
print(parit_2023[["a", "b", "pi", "sub_lo", "sub_hi"]].round(4).tail(5).to_string(index=False))

Kauden 2023–27 eniten ja vähiten eroavat parit:
   a   b     pi  sub_lo  sub_hi
 VAS RKP 0.5685  0.5540  0.5722
 VAS  PS 0.5570  0.5512  0.5615
 VAS KOK 0.5553  0.5494  0.5574
VIHR  PS 0.5529  0.5502  0.5558
 SDP RKP 0.5506  0.5378  0.5510
...
   a   b     pi  sub_lo  sub_hi
KESK  KD 0.5141  0.5023  0.5195
 KOK  PS 0.5137  0.5114  0.5163
 RKP  KD 0.5014  0.4863  0.5017
  KD  PS 0.4990  0.4892  0.5093
  KD KOK 0.4898  0.4807  0.4973
